# 05 RSCC claims on predicted evidence

Member 4. Notebook 04 had only 39 claims from 10 tiles, because the Qwen3-VL captions of the 20 Florence and Nepal tiles are mostly "no buildings". This notebook uses the RSCC captions instead: every Florence and Nepal crop (2,184 and 2,476) with a clear caption gives one tile-level claim, "none of the buildings are damaged" or "most are damaged".

**What it tests.** RSCC claims are easy and come from xBD labels, so against the label evidence most are supported by construction. The question is how many of them the checker still supports when the evidence comes from the CV model. That measures the vision model's damage grading through the chain, not the captioner.

1. Extract the claims and stage the evidence files on local disk.
2. Check every claim against label evidence and predicted evidence.
3. Pass-through, false support, by disaster and by claim type; a relaxed sensitivity check.
4. Confidence, cloud, bootstrap over parent images.
5. Save.

Runs on CPU. Nepal is the test event: nothing is tuned on it, and the two disasters are reported separately. Clear outputs before committing. Never commit evidence, captions or imagery.

In [ ]:
BRANCH = "main"

import os, subprocess, sys

REPO_DIR = "/content/FloodVerify"
if "google.colab" in sys.modules:
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "https://github.com/aaarrvind/FloodVerify.git", REPO_DIR], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "matplotlib"], check=True)
    REPO = REPO_DIR
else:
    REPO = os.path.abspath("../..")
sys.path.insert(0, os.path.join(REPO, "src"))
print("repo:", REPO)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

PROJECT = "/content/drive/MyDrive/MAAI 902 X 901 PROJECT"
ROOT = f"{PROJECT}/xbd"
LABELS_SRC = f"{ROOT}/processed/evidence_labels"                       # label evidence on Drive
PRED_SRC = f"{PROJECT}/evidence_predicted/m1_siamese_unet_ce_s0"       # vision team, read only
LABELS_DIR = "/content/ev_labels_rscc"                                 # local copies of the files we need
PRED_DIR = "/content/ev_pred_rscc"
MANIFEST = f"{ROOT}/rscc_flood_manifest.csv"
CLOUD_CSV = f"{PROJECT}/eval_extras/cloud_screen_val_test.csv"
OUT = f"{ROOT}/processed/captions/predicted_check"                     # our own folder
os.makedirs(OUT, exist_ok=True)

EVENTS = ["hurricane-florence", "nepal-flooding"]    # the two events with predicted evidence
TUNING_EVENT, TEST_EVENT = "hurricane-florence", "nepal-flooding"
CONF_CUT = 0.7
SEED = 42

## 1. Claims from the RSCC captions

Extraction is now in the repo (`floodsitrep.nlp.rscc`), the same rule as notebook 03: a caption that clearly says all (or both, or the single) buildings are undamaged gives "none damaged", and one that says all buildings have major damage, severe damage or are destroyed gives "most damaged". Captions with unknown or obscured buildings, partly damaged buildings or no buildings are skipped.

In [ ]:
import re, json, shutil, collections
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from floodsitrep.nlp.checker import bin_of, check_claim, class_fraction, MIN_VALID
from floodsitrep.nlp.rscc import rscc_tile_quantity
from floodsitrep.nlp.schema import Claim

manifest = pd.read_csv(MANIFEST, dtype={"tile_id": str, "part": str})
manifest["tile"] = manifest["event"] + "_" + manifest["tile_id"] + "_p" + manifest["part"]
manifest["parent"] = manifest["event"] + "_" + manifest["tile_id"]          # the 1024 px image the crop comes from
manifest = manifest[manifest["event"].isin(EVENTS)].reset_index(drop=True)
print(manifest.shape, manifest["event"].value_counts().to_dict())

In [ ]:
# The extraction rule lives in floodsitrep.nlp.rscc (tested in tests/nlp/test_rscc.py).
rscc_claims = manifest.assign(quantity=manifest["caption"].map(rscc_tile_quantity)).dropna(subset=["quantity"]).reset_index(drop=True)
print(len(rscc_claims), "claims from", len(manifest), "captions")
print(rscc_claims.groupby(["event", "quantity"]).size())

## 2. Stage the evidence files

Reading thousands of small JSON files from Drive is slow (the CV team measured 10 to 25 minutes for the whole folder). Only the files for crops that have a claim are copied, with several threads.

In [ ]:
def stage_files(names, src, dst, workers=16):
    """Copy <name>.json from src to dst (skips files already there). Returns a Counter of copied/have/missing."""
    if os.path.abspath(src) == os.path.abspath(dst):
        return collections.Counter(("have" if os.path.exists(f"{src}/{n}.json") else "missing") for n in names)
    os.makedirs(dst, exist_ok=True)

    def one(n):
        s, d = f"{src}/{n}.json", f"{dst}/{n}.json"
        if os.path.exists(d):
            return "have"
        if not os.path.exists(s):
            return "missing"
        shutil.copy(s, d)
        return "copied"

    with ThreadPoolExecutor(workers) as ex:
        return collections.Counter(ex.map(one, names))


tiles = list(rscc_claims["tile"])
print("labels   :", dict(stage_files(tiles, LABELS_SRC, LABELS_DIR)))
print("predicted:", dict(stage_files(tiles, PRED_SRC, PRED_DIR)))
ok = [t for t in tiles if os.path.exists(f"{LABELS_DIR}/{t}.json") and os.path.exists(f"{PRED_DIR}/{t}.json")]
claims = rscc_claims[rscc_claims["tile"].isin(ok)].reset_index(drop=True)
print(len(claims), "claims with both evidences")
print(claims.groupby(["event", "quantity"]).size())

## 3. The check on both evidences

The strict check is now the repo checker itself (`check_claim` with `cell=None`), which applies the same rule as notebook 03: the damaged share is the building-weighted share over the usable cells (buildings > 0 and `valid_fraction` at least 0.5), and the claim is supported when `bin_of(share)` equals the claimed word. So "none" needs no damaged building at all, and "most" needs at least 60%.

The **relaxed** check is a sensitivity analysis fixed before looking at the results: "none" is supported when the share is below 10% (bins none or few) and "most" when it is at least 30% (bins many or most). It shows how much of the loss is the strict exact-bin rule.

In [ ]:
def tile_stats(evidence):
    """Building-weighted damaged share, building count and mean confidence over the usable cells of one evidence file."""
    usable = [c for c in evidence["cells"] if c["buildings"] > 0 and c.get("valid_fraction", 1.0) >= MIN_VALID]
    total = sum(c["buildings"] for c in usable)
    if total == 0:
        return {"share": None, "buildings": 0, "conf": None}
    damaged = sum(c["buildings"] * (class_fraction([c], "damaged") or 0.0) for c in usable)
    conf = sum(c["buildings"] * c.get("confidence", 0.0) for c in usable) / total
    return {"share": damaged / total, "buildings": total, "conf": conf}


def verdict_from_share(quantity, share, relaxed=False):
    """supported / contradicted / not_verifiable for a whole-tile claim of 'none' or 'most'."""
    if share is None:
        return "not_verifiable"
    if relaxed:
        ok = share < 0.10 if quantity == "none" else share >= 0.30
    else:
        ok = bin_of(share) == quantity
    return "supported" if ok else "contradicted"


def verdict_of(quantity, evidence, share, relaxed=False):
    """Strict: the repo checker on a whole-image claim. Relaxed: the sensitivity rule on the damaged share."""
    if relaxed:
        return verdict_from_share(quantity, share, relaxed=True)
    return check_claim(Claim(text="", cell=None, quantity=quantity, damage_class="damaged"), evidence)


def compare_rscc(claims, labels_dir, pred_dir, relaxed=False):
    """One row per claim: the same claim checked against label evidence and predicted evidence."""
    rows = []
    for _, r in claims.iterrows():
        ev_a = json.load(open(f"{labels_dir}/{r['tile']}.json"))
        ev_b = json.load(open(f"{pred_dir}/{r['tile']}.json"))
        a, b = tile_stats(ev_a), tile_stats(ev_b)
        rows.append({"tile": r["tile"], "parent": r["parent"], "event": r["event"], "quantity": r["quantity"],
                     "bldg_labels": a["buildings"], "bldg_pred": b["buildings"],
                     "share_labels": a["share"], "share_pred": b["share"], "conf_pred": b["conf"],
                     "verdict_labels": verdict_of(r["quantity"], ev_a, a["share"], relaxed),
                     "verdict_pred": verdict_of(r["quantity"], ev_b, b["share"], relaxed)})
    return pd.DataFrame(rows)


VERDICTS = ["supported", "contradicted", "not_verifiable"]
cmp = compare_rscc(claims, LABELS_DIR, PRED_DIR)
cmp_relaxed = compare_rscc(claims, LABELS_DIR, PRED_DIR, relaxed=True)
print(len(cmp), "claims")
print(pd.crosstab(cmp["verdict_labels"], cmp["verdict_pred"]).reindex(index=VERDICTS, columns=VERDICTS, fill_value=0))
print("verdict unchanged:", round(float((cmp["verdict_labels"] == cmp["verdict_pred"]).mean()), 3))

In [ ]:
def passthrough_table(cmp, by=("event", "quantity")):
    """Per group: how many label-supported claims stay supported, and how many label-contradicted claims turn supported."""
    rows = []
    groups = [("all", cmp)] + [(" / ".join(map(str, k if isinstance(k, tuple) else (k,))), g) for k, g in cmp.groupby(list(by))]
    for name, g in groups:
        ls = g[g["verdict_labels"] == "supported"]
        lc = g[g["verdict_labels"] == "contradicted"]
        rows.append({"group": name, "claims": len(g),
                     "labels_supported": len(ls),
                     "pass_through": round(float((ls["verdict_pred"] == "supported").mean()), 3) if len(ls) else None,
                     "lost_to_contradicted": int((ls["verdict_pred"] == "contradicted").sum()),
                     "lost_to_not_verifiable": int((ls["verdict_pred"] == "not_verifiable").sum()),
                     "labels_contradicted": len(lc),
                     "false_support": round(float((lc["verdict_pred"] == "supported").mean()), 3) if len(lc) else None,
                     "verdict_unchanged": round(float((g["verdict_labels"] == g["verdict_pred"]).mean()), 3),
                     "pred_not_verifiable": round(float((g["verdict_pred"] == "not_verifiable").mean()), 3)})
    return pd.DataFrame(rows).set_index("group")


print("STRICT (exact bin)")
tab_strict = passthrough_table(cmp)
print(tab_strict.to_string())
print()
print("RELAXED (none: share < 10%, most: share >= 30%)")
tab_relaxed = passthrough_table(cmp_relaxed)
print(tab_relaxed.to_string())

How to read it:

- `pass_through` is the share of claims that are supported against the labels and still supported against the predicted evidence. It is the chain's ability to confirm a correct claim, and it is the number the 39-claim run could not give.
- `lost_to_contradicted` is the vision model's damage error reaching the checker. `lost_to_not_verifiable` means the prediction has no usable buildings in the tile.
- `false_support` is the share of claims that are contradicted against the labels and supported against the prediction. Agreement that is not evidence of correctness.
- Compare the strict and relaxed tables. If the relaxed pass-through is much higher, the loss comes mainly from the exact-bin rule meeting damage grades that are one level off (the CV team reports this for no damage / minor and minor / major).

In [ ]:
def plot_share_hist(cmp):
    """Predicted damaged share for the claims that the labels support, by claim type and disaster."""
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), sharey=False)
    for ax, q in zip(axes, ["none", "most"]):
        g = cmp[(cmp["quantity"] == q) & (cmp["verdict_labels"] == "supported") & cmp["share_pred"].notna()]
        for ev in EVENTS:
            ax.hist(g.loc[g["event"] == ev, "share_pred"], bins=np.linspace(0, 1, 11), alpha=0.6, label=ev)
        ax.axvline(0.10 if q == "none" else 0.60, color="grey", lw=1, ls="--")
        ax.set_title(f'claim "{q}": predicted damaged share')
        ax.set_xlabel("predicted damaged share of buildings"); ax.set_ylabel("claims")
        ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()


plot_share_hist(cmp)

In [ ]:
def plot_passthrough(tab):
    """Pass-through by disaster and claim type, strict and relaxed side by side."""
    import matplotlib.pyplot as plt
    names = [n for n in tab["strict"].index if n != "all"]
    x = np.arange(len(names))
    fig, ax = plt.subplots(figsize=(7, 3.6))
    ax.bar(x - 0.2, tab["strict"].loc[names, "pass_through"].astype(float), 0.4, label="strict (exact bin)")
    ax.bar(x + 0.2, tab["relaxed"].loc[names, "pass_through"].astype(float), 0.4, label="relaxed")
    ax.set_xticks(x); ax.set_xticklabels(names, rotation=15)
    ax.set_ylim(0, 1); ax.set_ylabel("label-supported claims still supported")
    ax.legend(); plt.tight_layout(); plt.show()


plot_passthrough({"strict": tab_strict, "relaxed": tab_relaxed})

### Why claims are lost

Each label-supported claim that is not supported against the prediction gets one reason from the building counts and shares.

In [ ]:
def loss_reasons(cmp):
    """Reasons for label-supported claims that the predicted evidence does not support."""
    lost = cmp[(cmp["verdict_labels"] == "supported") & (cmp["verdict_pred"] != "supported")].copy()

    def why(r):
        if r["verdict_pred"] == "not_verifiable":
            return "predicted evidence has no usable buildings in the tile"
        if r["quantity"] == "none":
            return "prediction finds damaged buildings where the labels have none"
        return "prediction finds a damaged share below 60% where the labels say most"

    lost["reason"] = lost.apply(why, axis=1)
    return lost.groupby(["event", "reason"]).size().rename("claims").reset_index().sort_values(["event", "claims"], ascending=[True, False])


loss_reasons(cmp)

In [ ]:
def building_totals(cmp):
    """Buildings in the claimed tiles, labels against prediction, per disaster (blob merging shows here)."""
    g = cmp.groupby("event")[["bldg_labels", "bldg_pred"]].sum()
    g["pred_over_labels"] = (g["bldg_pred"] / g["bldg_labels"]).round(2)
    return g


building_totals(cmp)

## 4. Confidence, cloud and uncertainty

- **Confidence.** Tile confidence is the building-weighted mean over usable cells. Below about 0.7 the CV team documents it as overconfident. The checker does not use it, so this is a breakdown only.
- **Cloud.** Crops with `cloud_post` above 0.10 are dropped in a second run.
- **Bootstrap.** The unit is the parent 1024 px image: the four crops of one image are not independent, so images are resampled, not claims.

In [ ]:
def by_confidence(cmp, cut=CONF_CUT):
    """Pass-through and verdict change split by the confidence of the predicted tile."""
    band = np.where(cmp["conf_pred"].isna(), "no predicted buildings",
                    np.where(cmp["conf_pred"] >= cut, f"confidence >= {cut}", f"confidence < {cut}"))
    return passthrough_table(cmp.assign(band=band), by=("band",))


by_confidence(cmp)

In [ ]:
def cloud_flagged(path, threshold=0.10):
    """Set of crop names with cloud_post above the threshold, or None when the file is missing."""
    if not os.path.exists(path):
        return None
    t = pd.read_csv(path)
    key = next((c for c in t.columns if c.lower() in ("tile", "tile_id", "crop", "name", "id", "file", "stem")), t.columns[0])
    names = t[key].astype(str).str.replace(r"\.(json|png)$", "", regex=True)
    return set(names[t["cloud_post"] > threshold])


flagged = cloud_flagged(CLOUD_CSV)
if flagged is None:
    print("no cloud screen file; skipping")
else:
    mine = cmp[cmp["tile"].isin(flagged)]
    print(len(flagged), "crops flagged,", len(mine), "of our claim tiles:")
    print(mine[["tile", "quantity", "verdict_labels", "verdict_pred"]].to_string())
    print("\nwithout cloud tiles:")
    print(passthrough_table(cmp[~cmp["tile"].isin(flagged)], by=("event",)).to_string())

In [ ]:
CELLS = [(a, b) for a in VERDICTS for b in VERDICTS]          # (verdict_labels, verdict_pred)


def parent_counts(cmp):
    """3x3 verdict-pair counts per parent image, as an array (parents x 9)."""
    c = cmp.groupby(["parent", "verdict_labels", "verdict_pred"]).size().unstack(["verdict_labels", "verdict_pred"], fill_value=0)
    return c.reindex(columns=pd.MultiIndex.from_tuples(CELLS), fill_value=0)


def bootstrap_chain(cmp, n_boot=2000, seed=SEED):
    """Resample parent images. Returns pass-through, false support, verdict unchanged and predicted not-verifiable share."""
    c = parent_counts(cmp)
    m = c.to_numpy()
    rng = np.random.default_rng(seed)
    draws = rng.integers(0, len(m), size=(n_boot, len(m)))
    s = m[draws].sum(axis=1)
    ix = {k: i for i, k in enumerate(CELLS)}

    def stats(s_):
        def col_(a, b):
            return s_[..., ix[(a, b)]]
        ls = sum(col_("supported", b) for b in VERDICTS)
        lc = sum(col_("contradicted", b) for b in VERDICTS)
        tot = s_.sum(axis=-1)
        return {"pass_through": col_("supported", "supported") / np.maximum(ls, 1),
                "false_support": col_("contradicted", "supported") / np.maximum(lc, 1),
                "verdict_unchanged": sum(col_(v, v) for v in VERDICTS) / np.maximum(tot, 1),
                "pred_not_verifiable": sum(col_(a, "not_verifiable") for a in VERDICTS) / np.maximum(tot, 1)}

    point = stats(m.sum(axis=0))
    boot = stats(s)
    rows = []
    for k in point:
        lo, hi = np.percentile(boot[k], [2.5, 97.5])
        rows.append({"metric": k, "value": round(float(point[k]), 3), "ci_low": round(float(lo), 3),
                     "ci_high": round(float(hi), 3), "parent_images": len(m)})
    return pd.DataFrame(rows)


boot_rows = []
for name, g in [("all", cmp)] + [(e, cmp[cmp["event"] == e]) for e in EVENTS]:
    b = bootstrap_chain(g)
    b.insert(0, "group", name)
    boot_rows.append(b)
boot = pd.concat(boot_rows, ignore_index=True)
boot

## 5. Save

In [ ]:
cmp.to_csv(f"{OUT}/rscc_labels_vs_predicted_strict.csv", index=False)
cmp_relaxed.to_csv(f"{OUT}/rscc_labels_vs_predicted_relaxed.csv", index=False)
tab_strict.to_csv(f"{OUT}/rscc_passthrough_strict.csv")
tab_relaxed.to_csv(f"{OUT}/rscc_passthrough_relaxed.csv")
boot.to_csv(f"{OUT}/rscc_bootstrap.csv", index=False)
print(sorted(os.listdir(OUT)))

## How to read this

- **Which evidence.** Every number says which evidence it uses. Label-evidence verdicts are the reference. Predicted-evidence verdicts show what the pipeline would report with the CV output.
- **What the claims are.** RSCC claims are easy ("none" or "most") and machine-written from xBD labels. High agreement against the labels is by construction, so this is a test of the vision model through the checker, not of a captioner.
- **Pass-through** (strict and relaxed) is the headline: how often a correct claim survives. **False support** is the other side: how often a wrong claim is confirmed.
- **Nepal is the test event.** Report it separately from Florence. The relaxed rule was fixed in advance and nothing was tuned on Nepal.
- **Blob merging.** Predicted counts are biased low in dense areas, much more in Nepal. Shares are ratios and less affected, but say which evidence each count comes from.
- **Independence.** Crops of one image are correlated; the intervals resample parent images. State the number of parent images next to every interval.